Stylistique numérique

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabays/32M7131/blob/master/Cours_08/Cours08.ipynb)

# Entraîner un classifieur avec _Spacy_

Simon Gabay

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licence Creative Commons" style="float: right" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a>

J'installe [Spacy](https://spacy.io/), une bibliothèque python pour le TAL très répandue et très bien documentée.
L'installation se fait avec `pip`, un gestionnaire de paquets utilisé pour installer et gérer des paquets écrits en Python. Pour l'utiliser, je dois faire la commande `pip install` + nom de la bibliothèque que je veux utiliser.
L'utilisation de `pip` n'est pas du python mais une [commande shell](https://doc.ubuntu-fr.org/commande_shell), je dois commencer la ligne avec `!`.

In [ ]:
!pip install spacy

Je vais maintenant pouvoir utiliser _Spacy_. Comme nous voulons utiliser un modèle de langue, je peux regarder [ce qui est proposé](https://spacy.io/models). Utilisons [le modèle multilingue](https://spacy.io/models/xx/) pour l'analyse d'entités nommées. Il est entraîné avec wikipedia.

In [48]:
!python -m spacy download xx_ent_wiki_sm

✔ Download and installation successful
You can now load the model via spacy.load('xx_ent_wiki_sm')


Maintenant je dois charger des données d'entraînement. Un petit document annoté est disponible en ligne sur le dépôt GitHub du cours:
1. Créons un dossier où le mettre avec la commande `mkdir` (_make directory_)
2. Téléchargeons le document avec `wget`, un programme en ligne de commande non interactif de téléchargement de fichiers depuis le Web, et l'URL du fichier que l'on veut récupérer (au format brut)

In [ ]:
!mkdir training_data
!wget https://raw.githubusercontent.com/gabays/32M7131/master/Cours_08/Data/BOYER_AGAMEMNON.tsv
!mv /content/BOYER_AGAMEMNON.tsv /content/training_data

Je peux maintenant regarder à quoi ressemble ce que j'ai téléchargé. Pour cela je vais avoir besoin de transformer le fichier `.tsv` en _dataframe_ et en afficher les premières lignes. La bibliothèque python qui permet de manipuler les _dataframes_, que je dois donc télécharger avec pip

In [14]:
!pip install pandas
# j'importe pandas sous le nom "pd" (comme cela c'est plus court, donc plus simple)
import pandas as pd

Je peux maintenant transformer mes données en _dataframe_ et en afficher les dix premières lignes

In [19]:
df = pd.read_csv('/content/training_data/BOYER_AGAMEMNON.tsv', sep="\t", header=None)
df.head(17)

,0,1
0,Oui,O
1,",",O
2,Pylade,O
3,",",O
4,il,O
5,est,O
6,vrai,O
7,",",O
8,la,O
9,valeur,O


Le fichier `.tsv` est impropre à être manipulé par _Spacy_ nous allons donc d'abord le transformer en `.json`. J'utilise une série de paramètres lors de cette conversion, notamment:
* `python -m spacy convert` pour la conversion
* Le fichier a traiter
* Le dossier ou stocker le résultat de la conversion
* `-t json` pour le format de sortie
* `-n 1` pour faire un document par phrae
* `-c iob` qui détecte que j'utilise une annotation qui suit le format standard _IOB_ (_inside, beginning, Outside_).

In [38]:
!mkdir training_json
!python -m spacy convert /content/training_data/BOYER_AGAMEMNON.tsv /content/training_json -t json -n 1 -c iob

mkdir: cannot create directory ‘training_json’: File exists
ℹ Auto-detected token-per-line NER format
ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (1002 documents):
/content/training_json/BOYER_AGAMEMNON.json


_Spacy_ nous indique qu'il recommande de regrouper les phrases par 10: suivons sa recommendation

In [39]:
!python -m spacy convert /content/training_data/BOYER_AGAMEMNON.tsv /content/training_json -t json -n 10 -c iob

ℹ Auto-detected token-per-line NER format
ℹ Grouping every 10 sentences into a document.
✔ Generated output file (101 documents):
/content/training_json/BOYER_AGAMEMNON.json


Affichons un petit extrait du résultat pour observer le résulat:

In [40]:
f = open('/content/training_json/BOYER_AGAMEMNON.json')
content = f.readlines()
print(*content[74:93])

                "orth":"Ont",
                 "tag":"-",
                 "ner":"O"
               },
               {
                 "orth":"de",
                 "tag":"-",
                 "ner":"O"
               },
               {
                 "orth":"l'",
                 "tag":"-",
                 "ner":"O"
               },
               {
                 "orth":"Asie",
                 "tag":"-",
                 "ner":"U-LOC"
               },



Après cette étape intermédiaire, je n'ai plus qu'à convertir mon fichier `.json` dans le format spécifique de _Spacy_.

In [54]:
!mkdir training_spacy
!python -m spacy convert /content/training_json/BOYER_AGAMEMNON.json /content/training_spacy -t spacy

mkdir: cannot create directory ‘training_spacy’: File exists

✘ Unknown file type: 'spacy'
Supported file types: 'json, jsonl, msg'



## On entraîne


In [56]:
!wget https://developer.nvidia.com/compute/cuda/9.2/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64 -O cuda-repo-ubuntu1604–9–2-local_9.2.88–1_amd64.deb
!dpkg -i cuda-repo-ubuntu1604–9–2-local_9.2.88–1_amd64.deb
!apt-key add /var/cuda-repo-9–2-local/7fa2af80.pub
!apt-get update
!apt-get install cuda-9.2
!nvcc --version

--2022-05-16 14:48:09--  https://developer.nvidia.com/compute/cuda/9.2/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64
Resolving developer.nvidia.com (developer.nvidia.com)... 152.195.19.142
Connecting to developer.nvidia.com (developer.nvidia.com)|152.195.19.142|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://developer.nvidia.com/compute/cuda/9.2/prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64 [following]
--2022-05-16 14:48:09--  https://developer.nvidia.com/compute/cuda/9.2/prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64
Reusing existing connection to developer.nvidia.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://developer.download.nvidia.com/compute/cuda/9.2/secure/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb?1EKca6KJJoZggYyiC-VbgwkzsfF00GfSwHckxba49Cp1Njgh2d5hiLvof2ZOSztaGZKAEnBwFV6Jjuzgm64Npobg4JC3-HSsejeQ5j_k_3uFd

Il faut désormais installer la bibliothèque _PyTorch_:

In [ ]:
pip install torch==1.7.1+cu92 torchvision==0.8.2+cu92 torchaudio==0.7.2 -f https://download.pytorch.org/whl/torch_stable.html